In [25]:
import pandas as pd

In [26]:
df = pd.read_csv("nsdi26ae.csv")

## Finding 1: The majority (52.2%) of the studied operator failures are caused by defects in operators’ interactions with external entities which significantly outnumbers bugs in operators’ internal program logic.

In [27]:
interaction_count = df[df["Root Cause Location"] == "Interaction"].shape[0]
total_count = df.shape[0]
percentage = (interaction_count / total_count * 100) if total_count > 0 else 0
print(f"The majority ({percentage:.1f}%) of the studied operator failures are "
      "caused by defects in operators' interactions with external entities "
      "which significantly outnumbers bugs in operators' internal program logic.")
# interaction_failures = df[df["Root Cause Location"] == "Interaction"]

The majority (52.2%) of the studied operator failures are caused by defects in operators' interactions with external entities which significantly outnumbers bugs in operators' internal program logic.


## Finding 3: Failures of managing applications is the largest category (42.3%) among all operator interaction failures.

In [28]:
interaction_failures = df[df["Root Cause Location"] == "Interaction"]
application_count = interaction_failures[interaction_failures["Interaction"] == "Application"]
application_percentage = (application_count.shape[0] / interaction_count * 100) if interaction_count > 0 else 0
print(f"Failures of managing applications is the largest category ({application_percentage:.1f}%) among all operator interaction failures.")

Failures of managing applications is the largest category (42.3%) among all operator interaction failures.


In [ ]:
interaction_failures = df[df["Root Cause Location"] == "Interaction"]
interaction_failures_by_operator = interaction_failures.groupby("Operator")["Interaction"].value_counts().unstack(fill_value=0)
interaction_failures_by_operator["Total"] = interaction_failures_by_operator.sum(axis=1)
interaction_failures_by_operator.loc["Total"] = interaction_failures_by_operator.sum()
interaction_failures_by_operator = interaction_failures_by_operator[["Application", "Co-operator", "Platform", "User", "Total"]]
interaction_failures_by_operator

Interaction,Application,Co-operator,Platform,User,Total
Operator,,,,,
CN/PostgresOp,15,0,1,4,20
CassOp,2,1,0,5,8
CockroachOp,4,0,6,2,12
KafkaOp,7,0,9,5,21
KnativeOp,0,1,4,8,13
KubeBlocks,12,0,7,1,20
MinIOOp,3,1,5,7,16
MongoOp,15,0,4,1,20
RabbitMQOp,3,1,10,0,14


## Finding 4: We find four main failure patterns:
- The majority (63.7%) of studied application-management failures are caused by the operator violating its managed application’s operation semantics.
- A significant percentage (16.5%) of management failures are caused by the gaps that prevent the operator from observing application internal states.
- Incompatibility between the operator and its managed application also caused a significant percentage (12.1%) of failures, triggered by upgrading application versions.
- The remaining cases (7.7%) were caused by the operator mishandling application errors.

In [36]:
# The majority (63.7%) of studied application-management failures are caused by the operator violating its managed application’s operation semantics.
semantic_violations = application_count[application_count["Pattern"] == "Semantic violations"]
semantic_violations_percentage = (semantic_violations.shape[0] / application_count.shape[0] * 100) if application_count.shape[0] > 0 else 0
print(f"The majority ({semantic_violations_percentage:.1f}%) of studied application-management failures are caused by the operator violating its managed application's operation semantics.")

# A significant percentage (16.5%) of management failures are caused by the gaps that prevent the operator from observing application internal states.
observation_gaps = application_count[application_count["Pattern"] == "State observability"]
observation_gaps_percentage = (observation_gaps.shape[0] / application_count.shape[0] * 100) if application_count.shape[0] > 0 else 0
print(f"A significant percentage ({observation_gaps_percentage:.1f}%) of management failures are caused by the gaps that prevent the operator from observing application internal states.")

# Incompatibility between the operator and its managed application also caused a significant percentage (12.1%) of failures, triggered by upgrading application versions.
incompatibility = application_count[application_count["Pattern"] == "Version incompatibility"]
incompatibility_percentage = (incompatibility.shape[0] / application_count.shape[0] * 100) if application_count.shape[0] > 0 else 0
print(f"Incompatibility between the operator and its managed application also caused a significant percentage ({incompatibility_percentage:.1f}%) of failures, triggered by upgrading application versions.")

# The remaining cases (7.7%) were caused by the operator mishandling application errors.
error_mishandling = application_count[application_count["Pattern"] == "Error handling"]
error_mishandling_percentage = (error_mishandling.shape[0] / application_count.shape[0] * 100) if application_count.shape[0] > 0 else 0
print(f"The remaining cases ({error_mishandling_percentage:.1f}%) were caused by the operator mishandling application errors.")

The majority (63.7%) of studied application-management failures are caused by the operator violating its managed application's operation semantics.
A significant percentage (16.5%) of management failures are caused by the gaps that prevent the operator from observing application internal states.
Incompatibility between the operator and its managed application also caused a significant percentage (12.1%) of failures, triggered by upgrading application versions.
The remaining cases (0.0%) were caused by the operator mishandling application errors.


In [40]:
# group by pattern and count occurrences
pattern_counts = application_count["Pattern"].value_counts()
pattern_counts

Pattern
Semantic violations               58
State observability               15
Version incompatibility           11
Mishandling application errors     7
Name: count, dtype: int64

## Table 5: The types of violated operation semantics.

In [42]:
operation_semantics = application_count[application_count["Pattern"] == "Semantic violations"]
operation_semantics_types = operation_semantics["Operation Semantics"].value_counts()
# display the total 
total_semantics = operation_semantics_types.sum()
operation_semantics_types = operation_semantics_types.reset_index()
operation_semantics_types.columns = ["Operation Semantics", "Count"]
operation_semantics_types["Percentage"] = (operation_semantics_types["Count"] / total_semantics * 100).round(1)
operation_semantics_types = operation_semantics_types.sort_values(by="Count", ascending=False).reset_index(drop=True)
print("Table 5: The types of violated operation semantics.")
print(operation_semantics_types)

Table 5: The types of violated operation semantics.
  Operation Semantics  Count  Percentage
0       Configuration     21        36.2
1            Ordering     18        31.0
2        Precondition     11        19.0
3         Environment      8        13.8
